# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya, using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset is described by a Croissant schema at the following URL:

https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json

In [ ]:
# Ensure the required library is installed
!pip install mlcroissant

## 1. Data Loading
Load the Croissant dataset and extract its metadata using the `mlcroissant` library.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset using mlcroissant
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets and their field/column `@id` values. All entities within Croissant (such as record sets, fields, columns) are referenced by their `@id` for reproducibility.
Let's enumerate the record sets and fields described in the metadata.

In [ ]:
# List record sets and their details by @id
record_sets = getattr(metadata, 'record_set', getattr(metadata, 'recordSets', []))
if not record_sets:  # If not found, try capitalized version (older schema)
    record_sets = getattr(metadata, 'RecordSet', [])

record_set_ids = []
if record_sets:
    for rs in record_sets:
        rec_id = getattr(rs, '@id', None) or getattr(rs, 'id', None)
        if rec_id is None: continue
        record_set_ids.append(rec_id)
        print(f"RecordSet @id: {rec_id}")
        print(f"  name: {getattr(rs, 'name', None)}")
        # List fields (by 'field' property, sometimes 'fields')
        fields = getattr(rs, 'field', getattr(rs, 'fields', []))
        for field in fields:
            print(f"    Field @id: {getattr(field, '@id', getattr(field, 'id', None))}")
            print(f"      name: {getattr(field, 'name', None)}")
            print(f"      dataType: {getattr(field, 'dataType', None)}")
        print()
else:
    print("No record sets found in metadata.")

## 3. Data Extraction
Let’s load the available record sets using the record set `@id` values identified above. We'll use these `@id` values—required by the Croissant model—so this will work with all FAIR datasets described by a Croissant schema.

In [ ]:
# If record set IDs are missing above (e.g. if they're not printed),
# you can inspect the dataset.metadata directly. For this dataset, we know from schema
# that the main record set is 'cr:MentalHealthSurveyKilifiRecordSet'.

# You may need to adjust the IDs/names if record sets/fields differ in a future schema version.
record_sets_to_load = [
    'cr:MentalHealthSurveyKilifiRecordSet'  # Replace with actual @id(s) from previous cell if different
]
dataframes = {}

for record_set_id in record_sets_to_load:
    try:
        records = list(dataset.records(record_set=record_set_id))
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded {len(records)} records for record set @id: {record_set_id}")
        print(f"Columns: {dataframes[record_set_id].columns.tolist()}")
    except Exception as e:
        print(f"Could not load record set {record_set_id}: {e}")

# Preview first record set loaded
if dataframes:
    first_rs = list(dataframes.keys())[0]
    dataframes[first_rs].head()

## 4. Exploratory Data Analysis (EDA)
Let’s perform EDA on the main record set. We’ll select and process some numeric and categorical fields by their `@id`.

Example numeric fields in this dataset: `'gad7_score'` (Generalized Anxiety Disorder 7-item score), `'phq9_score'`, `'psq_score'`.

Example group/categorical field: `'gender'`, `'village'`, `'level_of_education'` (use the actual field names or @ids available in your DataFrame).

In [ ]:
# Adjust these field names if needed according to the actual column names (@id or 'gad7_score', etc.)
main_record_set = 'cr:MentalHealthSurveyKilifiRecordSet'
df = dataframes.get(main_record_set)

# --- Numeric field analysis ---
numeric_field = 'gad7_score'  # Use the actual @id if available, otherwise column name.
if numeric_field in df.columns:
    threshold = 10  # GAD-7 suggested clinical cutoff
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Records with {numeric_field} > {threshold}: {len(filtered_df)}")
    print(filtered_df[[numeric_field]].head())

    # Normalize
    filtered_df[numeric_field + '_normalized'] = (
        filtered_df[numeric_field] - filtered_df[numeric_field].mean()
    ) / filtered_df[numeric_field].std()
    print(filtered_df[[numeric_field, numeric_field+'_normalized']].head())
else:
    print(f"Numeric field {numeric_field} not found in columns: {df.columns.tolist()}")

# --- Group by a categorical field ---
group_field = 'gender'  # Use the actual @id if present ('gender', 'village', etc.)
if group_field in df.columns:
    grouped = df.groupby(group_field)[numeric_field].mean()
    print(f"\nAverage {numeric_field} by {group_field}:")
    print(grouped)
else:
    print(f"Group field {group_field} not in dataframe columns: {df.columns.tolist()}")

## 5. Visualization
Let’s visualize the distribution of the `gad7_score` (or other mental health indicators) and their breakdown by group, such as gender.

If you want to visualize additional fields, adjust the following code to use the appropriate `@id` or column name.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df is not None and 'gad7_score' in df.columns:
    plt.figure(figsize=(7,4))
    sns.histplot(df['gad7_score'].dropna(), bins=15, kde=True)
    plt.title("Distribution of GAD-7 Scores")
    plt.xlabel("GAD-7 Score")
    plt.ylabel("Number of participants")
    plt.show()

    # Boxplot by gender
    if 'gender' in df.columns:
        plt.figure(figsize=(6,4))
        sns.boxplot(data=df, x='gender', y='gad7_score')
        plt.title("GAD-7 Score by Gender")
        plt.xlabel("Gender")
        plt.ylabel("GAD-7 Score")
        plt.show()

## 6. Conclusion
- The FAIR² Mental Health Survey Dataset from Kilifi County provides rich demographic and mental health assessment data suitable for research and AI-ready analyses.
- Using the Croissant schema model, all dataset entities are referenced by their `@id`, ensuring consistency and reproducibility.
- Initial EDA reveals the overall distribution and group differences in mental health indicators such as the GAD-7 score.

Continue your analysis by exploring additional fields, records sets, or visualizations. For full schema details and metadata, consult the Croissant schema JSON-LD at the dataset source URL.